# EcoData · S3 · Calidad de datos y Parquet particionado · Dimensión Aire (Lanzeloth)
Nulos, duplicados (Window + row_number), rangos, huecos, Silver → Gold por mes y verificación con PartitionFilters. Requiere haber corrido S2.

In [1]:
import os, json, time, getpass, platform
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
spark=(SparkSession.builder.master("local[2]").appName("EcoData-U1-Aire")
       .config("spark.ui.enabled","false").config("spark.sql.session.timeZone","UTC").config("spark.sql.shuffle.partitions","4").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Evidencia | usuario:",getpass.getuser(),"| equipo:",platform.node(),"| reloj:",time.strftime("%Y-%m-%d %H:%M:%S"))
print("Spark",spark.version)

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=34977 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=
Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=34977 -Dhttp.nonProxyHosts=localhost|127.0.0.1

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 20:24:34 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/11/18 20:24:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 20:24:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Evidencia | usuario: root | equipo: vm | reloj: 2026-11-18 20:24:37
Spark 4.0.1


In [2]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
esquema=StructType([StructField("time",StringType(),True),StructField("pm10",DoubleType(),True),StructField("pm2_5",DoubleType(),True)])
bronze=spark.read.csv("bronze/aire_juliaca_2023.csv",header=True,schema=esquema)   # lo genera el notebook S2
print("Bronze filas:",bronze.count())

Bronze filas: 8760


In [3]:
req={"time","pm10","pm2_5"}; assert req<=set(bronze.columns),"faltan columnas"
n0=bronze.count()
nulos=bronze.select([F.sum(((F.col(c).isNull())|(F.col(c).cast("string")==""))  .cast("int")).alias(c) for c in bronze.columns]).first().asDict()
dups=n0-bronze.dropDuplicates(["time"]).count()
rango=bronze.filter((F.col("pm2_5")<0)|(F.col("pm10")<0)).count(); redond=bronze.filter(F.col("pm2_5")>F.col("pm10")).count(); print("pm2_5>pm10 por redondeo (se conservan):",redond)
print("ANTES  filas=%d | nulos=%s | duplicados(time)=%d | valores negativos=%d"%(n0,nulos,dups,rango))
w=Window.partitionBy("time").orderBy(F.col("pm2_5").desc_nulls_last())      # deduplicación determinista
silver=(bronze.dropna(subset=["pm10","pm2_5"]).withColumn("rn",F.row_number().over(w)).filter("rn=1").drop("rn")
        .filter((F.col("pm2_5")>=0)&(F.col("pm10")>=0))   # solo valores negativos son inválidos; pm10<pm2_5 por redondeo se conserva (se reporta)
        .withColumn("ts",F.to_timestamp("time")).withColumn("mes",F.date_format("ts","yyyy-MM")))
n1=silver.count()
print("DESPUÉS filas=%d | descartadas=%d"%(n1,n0-n1))
# huecos horarios
tt=silver.orderBy("ts").select(F.col("ts").cast("long").alias("s")).toPandas().s.values
print("huecos horarios (>1 h entre registros):",int((np.diff(tt)>3600).sum()))

pm2_5>pm10 por redondeo (se conservan): 12
ANTES  filas=8760 | nulos={'time': 0, 'pm10': 0, 'pm2_5': 0} | duplicados(time)=0 | valores negativos=0


DESPUÉS filas=8760 | descartadas=0


huecos horarios (>1 h entre registros): 0


In [4]:
RUTA="gold/aire_mensual"
(silver.select("ts","pm10","pm2_5","mes").write.mode("overwrite").partitionBy("mes").parquet(RUTA))
back=spark.read.parquet(RUTA)
assert back.count()==n1, "la lectura de vuelta no coincide"
print("round-trip OK:",back.count(),"filas |",len([d for d in os.listdir(RUTA) if d.startswith("mes=")]),"particiones")
print(sorted(d for d in os.listdir(RUTA) if d.startswith("mes="))[:4],"...")
back.filter(F.col("mes")=="2023-06").explain()   # buscar PartitionFilters

round-trip OK: 8760 filas | 12 particiones
['mes=2023-01', 'mes=2023-02', 'mes=2023-03', 'mes=2023-04'] ...
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [ts#90,pm10#91,pm2_5#92,mes#93] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/claude-0/-home-claude/89735d0e-216f-51a3-b4cf-e9505671be82/s..., PartitionFilters: [isnotnull(mes#93), (mes#93 = 2023-06)], PushedFilters: [], ReadSchema: struct<ts:timestamp,pm10:double,pm2_5:double>




In [5]:
import io, contextlib
buf=io.StringIO()
with contextlib.redirect_stdout(buf): back.filter(F.col("mes")=="2023-06").explain()
print([l.strip() for l in buf.getvalue().splitlines() if "PartitionFilters" in l])

['+- FileScan parquet [ts#90,pm10#91,pm2_5#92,mes#93] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/claude-0/-home-claude/89735d0e-216f-51a3-b4cf-e9505671be82/s..., PartitionFilters: [isnotnull(mes#93), (mes#93 = 2023-06)], PushedFilters: [], ReadSchema: struct<ts:timestamp,pm10:double,pm2_5:double>']
